# Quality setting vs fidelity in the DCT codec

Scope: the transform-and-quantisation stage only. No entropy coding, no chroma subsampling. This is not a
JPEG encoder.

Session plan: state the fidelity target, audit the metric the code prints, prove the transform is exact,
prove the corrected metric, sweep both images, document the settings that behave invalidly, then decide.

Cells marked WRITE IT are empty on purpose. Helper cells are pre-built plumbing, not findings.


In [1]:
import importlib.util, numpy as np, matplotlib.pyplot as plt
np.set_printoptions(precision=2, suppress=True, linewidth=160)


## 1. The existing implementation

The basis, the transform and the quantisation table come from `DCT.py` in this repository, unmodified.

In [2]:
spec = importlib.util.spec_from_file_location("dctmod", "DCT.py")
dctmod = importlib.util.module_from_spec(spec); spec.loader.exec_module(dctmod)
T = dctmod.dct_coeff(); Tpinv = np.linalg.inv(T)
print("T:", T.shape)


T: (8, 8)


## 2. Image in, reconstruction out (plumbing)

`reconstruct` applies the repository's blockwise transform, quantisation and inverse. Blocks whose size
is not a multiple of eight are left as they were initialised, exactly as the repository does.

In [3]:
def load(path):
    import cv2
    I = cv2.imread(path)
    return I.astype(float)

def reconstruct(img, level):
    Q = dctmod.quantization_level(level)
    out = np.zeros_like(img)
    H, W, _ = img.shape
    for c in range(3):
        ch = img[:, :, c] - 128.0
        with np.errstate(divide="ignore", invalid="ignore"):
            for i in range(H // 8):
                for j in range(W // 8):
                    blk = ch[8*i:8*i+8, 8*j:8*j+8]
                    coef = T @ blk @ Tpinv
                    q = np.round(coef / Q)
                    out[8*i:8*i+8, 8*j:8*j+8, c] = Tpinv @ (q * Q) @ T
    out = np.nan_to_num(out, nan=0.0, posinf=255.0, neginf=0.0) + 128.0
    return np.clip(np.round(out), 0.0, 255.0)   # the stored image is 8-bit   # the stored image is 8-bit, so clip before measuring


In [4]:
def nz_share(img, Q):
    """Share of quantised coefficients that survive, as a percentage, across all blocks and channels."""
    nz = tot = 0
    for c in range(img.shape[2]):
        ch = img[:, :, c] - 128.0
        for i in range(img.shape[0] // 8):
            for j in range(img.shape[1] // 8):
                q = np.round((T @ ch[8*i:8*i+8, 8*j:8*j+8] @ Tpinv) / Q)
                nz += int((q != 0).sum()); tot += 64
    return 100.0 * nz / tot


## 3. The fidelity target

State it before measuring anything. A threshold chosen after seeing results is a rationalisation.

TARGET: _(type it here — the number, the metric, and why that number is defensible)_

## 4. Test content

Generated deterministically. A block with no high-frequency content cannot show what quantisation removes.

In [ ]:
yy, xx = np.mgrid[0:8, 0:8]
smooth = np.clip(96 + 6.0*xx + 70.0*(xx >= 4) + 12.0*np.sin(np.pi*yy/8), 0, 255)
rng = np.random.default_rng(7)
texture = np.clip(70 + 55*(xx + yy >= 8) + 18*np.sin(1.7*xx + 0.9*yy) + rng.normal(0, 8, (8, 8)), 0, 255)
flat = np.clip(np.full((8, 8), 128.0) + 3.0*np.sin(np.pi*yy/8), 0, 255)
blocks = {"smooth": np.round(smooth), "texture": np.round(texture), "flat": np.round(flat)}
fig, ax = plt.subplots(1, 3, figsize=(9, 3))
for a, (k, b) in zip(ax, blocks.items()):
    a.imshow(b, cmap="gray", vmin=0, vmax=255); a.set_title(k); a.axis("off")
plt.show()


## 5. The quality knob

`dctmod.quantization_level(n)` maps a level to an 8x8 step table. Reproduce its rule, then inspect the
extremes.

In [ ]:
Q50 = np.array([[16,11,10,16,24,40, 51, 61],
                [12,12,14,19,26,58,60,55],
                [14,13,16,24,40,57,69,56],
                [14,17,22,29,51,87,80,62],
                [18,22,37,56,68,109,103,77],
                [24,35,55,64,81,104,113,92],
                [49,64,78,87,103,121,120,101],
                [72,92,95,98,112,100,103,99]], dtype=float)

def quant_table(q):
    # WRITE IT (about 4 lines): map a quality level q to an 8x8 step table from Q50.
    raise NotImplementedError("quality-to-step-size rule not implemented yet")

# then check it reproduces the code's own table for one level:
#   np.array_equal(quant_table(50), dctmod.quantization_level(50))


## 6. One block, through the pipeline

The transform is easier to trust when one block's journey is visible: pixels, coefficients, quantised
coefficients, then the reconstruction. Everything after this just does the same thing 32,000 times.


In [ ]:
img = load("Image/scene_a.jpg")
print("image:", img.shape)


In [ ]:
blk_i, blk_j = 528, 944          # an 8x8 block on the number plate
blk = img[blk_i:blk_i+8, blk_j:blk_j+8, 0] - 128.0
Q = quant_table(50)
print("the block, shifted by 128:")
print(blk.astype(int))


## 6a. Where the basis comes from

$T$ is not a lookup table. It is the 8-point DCT-II basis, and the 2-D transform applies it separably —
across the rows, then down the columns.

$$C[k,n] \;=\; \alpha(k)\,\cos\!\left(\frac{\pi\,(2n+1)\,k}{16}\right),
\qquad \alpha(0)=\sqrt{\tfrac{1}{8}},\quad \alpha(k)=\sqrt{\tfrac{2}{8}}\ \ \text{for } k>0$$

Forward transform of an 8x8 block $X$: $\;F = T\,X\,T^{\mathsf T}$.

Inverse: $\;X = T^{\mathsf T}F\,T$, which is only valid because $T$ is orthonormal, $T\,T^{\mathsf T}=I$.

Quantisation, the single lossy step: $\;F^{q}_{uv} = \operatorname{round}\!\left(F_{uv}\,/\,Q_{uv}\right)$.


In [ ]:
k = np.arange(8)[:, None]; n = np.arange(8)[None, :]
alpha = np.where(k == 0, 1 / np.sqrt(8), np.sqrt(2 / 8))
C = alpha * np.cos(np.pi * (2 * n + 1) * k / 16)
print("basis rebuilt from the formula matches the repo's matrix:", np.allclose(C, T))


In [ ]:
# WRITE IT (1 line): transform the block with the basis on both sides, and print the coefficients


In [ ]:
# WRITE IT (2 lines): divide by the step table, round, print the result, and count how many of the 64 survive


In [ ]:
# WRITE IT (2 lines): multiply back by the step table, invert the transform, and report the largest
# pixel error in this block


In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(11, 3))
ax[0].imshow(blk + 128, cmap="gray", vmin=0, vmax=255); ax[0].set_title("pixels")
ax[1].imshow(np.abs(coef), cmap="viridis"); ax[1].set_title("|coefficients|")
ax[2].imshow(q, cmap="viridis"); ax[2].set_title("quantised")
ax[3].imshow(back + 128, cmap="gray", vmin=0, vmax=255); ax[3].set_title("reconstructed")
for a in ax: a.axis("off")
plt.show()


## 7. Measure the fidelity — RMSE and PSNR at this setting

Two numbers, computed on the reconstruction against the original. RMSE is in grey levels of a 0 to 255
range; PSNR expresses that error against the peak, in decibels, so it is comparable across images.


In [ ]:
mid = reconstruct(img, 50)
print("reconstruction error present:", not np.allclose(img, mid))


In [ ]:
# WRITE IT (about 5 lines): rmse = sqrt(mean((reconstruction - original) ** 2)) over all channels,
#   then psnr = 10 * log10(255 ** 2 / mse); print both, rounded to 4 and 2 decimals


## 8. The transform itself is innocent

In [ ]:
# WRITE IT (1-2 lines): check T @ T.T == I and print the largest deviation


In [ ]:
# WRITE IT (2 lines): one block, transform then inverse with no quantisation, report max abs error


## 9. Sweep the knob — image A

In [ ]:
LEVELS = [90, 70, 50, 40, 30, 20, 10]
# WRITE IT (about 5 lines): for each level, build the table, reconstruct, then collect
# step range, nz_share, rmse, psnr and max abs error into rows, printing each.


## 10. Reproduce on image B

In [ ]:
# WRITE IT (about 3 lines): same sweep, second image, same output columns


## 11. Settings whose behaviour is invalid

Document these rather than reporting them as results.

In [ ]:
# WRITE IT (2 lines): the top of the range, and what the step table contains there


In [ ]:
# WRITE IT (2 lines): the coarse end, and whether the table keeps discriminating


In [ ]:
# WRITE IT (about 3 lines): an image whose dimensions are not a multiple of eight


## Comparison table

| level | step min-max | non-zero coeffs | RMSE | PSNR (dB) | max abs error | meets target |
|---|---|---|---|---|---|---|
| | | | | | | |


## Decision and teaching conclusion

_Unfinished: the target, the lowest setting that holds it on the harder image, where the error concentrates,
and what that setting costs visually._

In [ ]:
# WRITE IT (about 3 lines): reprint the whole table from the variables, so the last frame is generated, not typed
